In [ ]:
# imports
import numpy as np
# import sys
import scipy.stats as stats

# sys.path.append('helpers/pcca_fa')
# import helpers.pcca_fa.pcca_fa_mdl as pf
from dual_pfc_funcs import load_dict, getParams, gen_GP, jitter

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

data_path = 'preprocessed_data/'
params = getParams()
subjects = params['subjects']
plot_sym = params['markers']

In [ ]:
# generate 2 GP with slow timescale component
gp_tau,gp_len = 50, 500
gp1 = gen_GP(50,gp_len,noise_var=3e-2,seed=5)
gp2 = gen_GP(50,gp_len,noise_var=3e-2,seed=2)
corr_val = np.corrcoef(gp1,gp2)[0,1]

# uncorrelate GP by removing slow component
# remove mean:
gp1_nomean = gp1 - np.mean(gp1)
gp2_nomean = gp2 - np.mean(gp2)

# remove slow component:
ar_order = 25
ar1, ar2 = np.zeros_like(gp1_nomean), np.zeros_like(gp2_nomean)
for i in range(gp_len):
    offset = int(ar_order / 2)
    start_idx = np.max((0, i-offset)) # incorporate boundary effects
    # perform full convolution
    ar1[i] = np.mean(gp1_nomean[start_idx:i+offset+1])
    ar2[i] = np.mean(gp2_nomean[start_idx:i+offset+1])
gp1_noar = gp1_nomean - ar1
gp2_noar = gp2_nomean - ar2

uncorr_val = np.corrcoef(gp1_noar.squeeze(),gp2_noar.squeeze())[0,1]

fig,ax = plt.subplots(1, 2, sharex=True, figsize=(10,4))
fig.tight_layout(pad=6.0)

ax[0].plot(gp1,color='0.6')
ax[0].plot(gp2,color='0.3')
ax[0].set_title('raw activity')
ax[0].annotate(r'$\rho={:.3f}$'.format(corr_val), xy=(25, 2),fontsize=12) 
ax[0].set_xlabel('trial number')
ax[0].set_ylabel('simulated activity (a.u.)')

ax[1].plot(gp1_noar,color='0.6')
ax[1].plot(gp2_noar,color='0.3')

ax[1].set_title('raw - slow component')
ax[1].annotate(r'$\rho={:.3f}$'.format(uncorr_val), xy=(25, 2),fontsize=12) 
ax[1].set_xlabel('trial number')
ax[1].set_ylabel('simulated activity (a.u.)')
ax[1].set_ylim(ax[0].get_ylim())

if SAVE_FIG:
    pdf = PdfPages('figs/ex_corr_gp.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# neural activity
dat = load_dict(data_path + 'fast_null_fits_0_25.pkl')

fig,ax = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10,4))
fig.tight_layout(pad=4.0)
fig.supxlabel('across-area co-fluctuation pattern')
fig.supylabel('cross-validated canonical correlation')

max_dim = 1
flag = True # determine when legend is shown
sub_idx_curr = -1
for i_sess,sess in enumerate(dat.keys()):
    sub_idx = np.where([sess[:2].lower() == sub[:2] for sub in subjects])[0][0]
    sym = plot_sym[sub_idx]
    if sub_idx > sub_idx_curr: 
        flag = True
    else:
        flag = False
    sub_idx_curr = sub_idx

    curr_dat = dat[sess]

    # # get CC for raw flip:
    # raw_mdl = pf.pcca_fa()
    # raw_mdl.set_params(curr_dat['raw_params'])
    # _,raw_rho = raw_mdl.get_canonical_directions()

    # # get CC for fast flip:
    # fast_mdl = pf.pcca_fa()
    # fast_mdl.set_params(curr_dat['fast_params'])
    # _,fast_rho = fast_mdl.get_canonical_directions()

    raw_rho = curr_dat['raw_rho']
    fast_rho = curr_dat['fast_rho']
    curr_dim = curr_dat['raw_params']['zDim']
    max_dim = np.max([max_dim,curr_dim])

    xdata = np.arange(curr_dim) + 1 + jitter(curr_dim,spacing=0.1)

    ax[0].plot(xdata,raw_rho,'-',color='gray',alpha=0.25,lw=1)
    if flag:
        ax[0].scatter(xdata,raw_rho,marker=sym,s=3,color='k',label='Monkey {:s}'.format(sess[:2].title()))
    else:
        ax[0].scatter(xdata,raw_rho,marker=sym,s=3,color='k')
    
    ax[1].plot(xdata,fast_rho,'-',color='gray',alpha=0.25,lw=1)
    ax[1].scatter(xdata,fast_rho,marker=sym,s=3,color='k')

ax[0].plot([1,14],[0,0],'k--')
ax[1].plot([1,14],[0,0],'k--')

ax[0].set_xticks(np.arange(0,max_dim,2)+2)
ax[0].set_ylim([-0.15,1])
ax[0].set_title('PFC: left vs right (shuffled) \n raw spike counts')
ax[1].set_title('PFC: left vs right (shuffled) \n raw - slow component')
ax[0].legend(loc='upper right',markerscale=2)

if SAVE_FIG:
    pdf = PdfPages('figs/corr_neural.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# slow component
dat = load_dict(data_path + 'slow_null_fits_0_25.pkl')

fig,ax = plt.subplots()
plt.xlabel('across-area co-fluctuation pattern')
plt.ylabel(r'$\rho_{slow} - \rho_{flip}$')

max_dim = 1
flag = True # determine when legend is shown
sub_idx_curr = -1
for i_sess,sess in enumerate(dat.keys()):
    sub_idx = np.where([sess[:2].lower() == sub[:2] for sub in subjects])[0][0]
    sym = plot_sym[sub_idx]
    if sub_idx > sub_idx_curr: 
        flag = True
    else:
        flag = False
    sub_idx_curr = sub_idx

    curr_dat = dat[sess]

    # # get CC true slow:
    # raw_mdl = pf.pcca_fa()
    # raw_mdl.set_params(curr_dat['slow_params'])
    # _,raw_rho = raw_mdl.get_canonical_directions()

    # # get CC for slow flip:
    # flip_mdl = pf.pcca_fa()
    # flip_mdl.set_params(curr_dat['slow_flip_params'])
    # _,flip_rho = flip_mdl.get_canonical_directions()

    raw_rho = curr_dat['slow_rho']
    flip_rho = curr_dat['slow_flip_rho']
    curr_dim = curr_dat['slow_params']['zDim']
    max_dim = np.max([max_dim,curr_dim])

    xdata = np.arange(curr_dim) + 1 + jitter(curr_dim,spacing=0.1)

    ax.plot(xdata,raw_rho-flip_rho,'-',color='gray',alpha=0.25,lw=1)
    if flag:
        ax.scatter(xdata,raw_rho - flip_rho,marker=sym,s=3,color='k',label='Monkey {:s}'.format(sess[:2].title()))
    else:
        ax.scatter(xdata,raw_rho - flip_rho,marker=sym,s=3,color='k')

ax.plot([1,14],[0,0],'k--')
ax.set_yticks(np.arange(-0.05,0.11,0.05))
ax.set_ylim([-0.06,0.11])
plt.title('PFC: left vs right \n slow component')
plt.legend(loc='upper right',markerscale=2)

if SAVE_FIG:
    pdf = PdfPages('figs/slow_corr_neural.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# slow component statistics:
alpha = 0.01 
print("Slow component statistics across sessions:")
for sub in subjects:
    print('Monkey:', sub[:2].title())
    
    curr_rho_slow = np.hstack([dat[sess]['slow_rho'] for sess in dat.keys() if sub[:2].title() in sess])
    curr_flip_rho = np.hstack([dat[sess]['slow_flip_rho'] for sess in dat.keys() if sub[:2].title() in sess])

    p = stats.ttest_rel(a=curr_rho_slow, b=curr_flip_rho,alternative='greater').pvalue
    print("  rho slow > rho flip? {}, p = {:.6f}, n = {} co-fluctuation patterns".format(p < alpha, p, curr_rho_slow.shape[0]))
    